In [ ]:
from pathlib import Path
from collections import defaultdict
import hashlib
import json
import pickle
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import partial_dependence
import shap

FOLDER = Path.cwd()
if not (FOLDER / 'Inputs' / 'RF_SHAP_samples.csv').is_file():
    FOLDER = FOLDER / 'Data Processing'
INPUT_DIR = FOLDER / 'Inputs'
OUTPUT_DIR = FOLDER / 'Outputs' / 'RF_SHAP'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SETTINGS = json.loads((INPUT_DIR / 'RF_SHAP_settings.json').read_text(encoding='utf-8'))
CLUSTER_THRESHOLD = 0.7
N_ESTIMATORS = 300
MODEL_SEED = 42
SAMPLING_SEED = 20261009
N_BOOTSTRAP = 500
N_JOBS = 1
SHAP_SAMPLE_SIZE = 800
N_DISPLAY = 12
STABILITY_THRESHOLD = 0.60

In [ ]:
samples = pd.read_csv(INPUT_DIR / 'RF_SHAP_samples.csv', float_precision='round_trip')
candidates = list(SETTINGS['predictor_columns'])
response_name = SETTINGS['response_column']
required = ['lat', 'lon', response_name] + candidates
missing = set(required) - set(samples.columns)
if missing:
    raise ValueError(f'Missing input columns: {sorted(missing)}')
samples = samples.loc[np.isfinite(samples[required].to_numpy(dtype=float)).all(axis=1)].reset_index(drop=True)
if len(samples) < 10:
    raise ValueError('At least ten complete samples are required.')
constant = [v for v in candidates if samples[v].nunique() < 2]
if constant:
    raise ValueError(f'Constant predictors: {constant}')
correlation = samples[candidates].corr(method='spearman')
if len(candidates) == 1:
    cluster_ids = np.ones(1, dtype=int)
else:
    distances = 1.0 - np.abs(correlation.to_numpy())
    np.fill_diagonal(distances, 0.0)
    tree = linkage(squareform(distances, checks=False), method='average')
    cluster_ids = fcluster(tree, 1.0 - CLUSTER_THRESHOLD, criterion='distance')
clusters = defaultdict(list)
for index, cluster_id in enumerate(cluster_ids):
    clusters[int(cluster_id)].append(index)
selected = []
cluster_rows = []
for cluster_id, indices in clusters.items():
    if len(indices) == 1:
        representative = indices[0]
    else:
        mean_correlations = [np.mean([abs(correlation.iloc[i, j]) for j in indices if j != i]) for i in indices]
        representative = indices[int(np.argmax(mean_correlations))]
    selected.append(candidates[representative])
    for index in indices:
        cluster_rows.append({'cluster': cluster_id, 'variable': candidates[index], 'selected': index == representative})
pd.DataFrame(cluster_rows).to_csv(OUTPUT_DIR / 'predictor_clusters.csv', index=False)
correlation.to_csv(OUTPUT_DIR / 'predictor_correlations.csv')
X = samples[selected].to_numpy()
y = samples[response_name].to_numpy()
print(f'Samples: {len(y)}; screened predictors: {len(candidates)}; cluster representatives: {len(selected)}')

In [ ]:
def fit_forest(features, response):
    model = RandomForestRegressor(n_estimators=N_ESTIMATORS, random_state=MODEL_SEED, n_jobs=N_JOBS)
    return model.fit(features, response)

model = fit_forest(X, y)
importances = model.feature_importances_
order = np.argsort(importances)[::-1]
rank = np.empty(len(selected), dtype=int)
rank[order] = np.arange(1, len(selected) + 1)
importance_table = pd.DataFrame({
    'rank': rank, 'variable': selected,
    'label': [SETTINGS['labels'].get(v, v) for v in selected],
    'importance': importances
}).sort_values('rank')
importance_table.to_csv(OUTPUT_DIR / 'importance.csv', index=False)
pd.DataFrame({'lat': samples.lat, 'lon': samples.lon, 'observed': y, 'fitted': model.predict(X)}).to_csv(OUTPUT_DIR / 'fitted_values.csv', index=False)
with (OUTPUT_DIR / 'random_forest.pkl').open('wb') as stream:
    pickle.dump(model, stream)
print(f'Training R2: {model.score(X, y):.6f}')

In [ ]:
sample_indices = np.random.RandomState(MODEL_SEED).choice(len(X), min(SHAP_SAMPLE_SIZE, len(X)), replace=False)
explanation_X = X[sample_indices]
explainer = shap.TreeExplainer(model)
shap_values = np.asarray(explainer.shap_values(explanation_X))
baseline = float(np.asarray(explainer.expected_value).reshape(-1)[0])
if not np.allclose(baseline + shap_values.sum(axis=1), model.predict(explanation_X), rtol=1e-6, atol=1e-8):
    raise AssertionError('SHAP contributions do not recover model predictions.')
display_indices = order[:min(N_DISPLAY, len(selected))]
display_names = [selected[i] for i in display_indices]
pd.DataFrame(shap_values[:, display_indices], columns=display_names).to_csv(OUTPUT_DIR / 'shap_values.csv', index=False)
pd.DataFrame(explanation_X[:, display_indices], columns=display_names).to_csv(OUTPUT_DIR / 'feature_values.csv', index=False)
pd.DataFrame(shap_values, columns=selected).to_csv(OUTPUT_DIR / 'shap_values_all.csv', index=False)
samples.loc[sample_indices, ['lat', 'lon']].to_csv(OUTPUT_DIR / 'shap_sample_coordinates.csv', index=False)
pdp = []
for index in display_indices:
    result = partial_dependence(model, X, features=[int(index)], grid_resolution=50)
    pdp.append({'average': np.asarray(result['average']).tolist(), 'grid_values': [np.asarray(v).tolist() for v in result['grid_values']]})
colors = ['#c1e7a4' if 'LAI' in v else '#f4a6a6' if 'CO2' in v else '#a8dadc' for v in display_names]
plot_settings = {'labels': [SETTINGS['labels'].get(v, v) for v in display_names], 'colors': colors, 'pdp': pdp}
(OUTPUT_DIR / 'plot_settings.json').write_text(json.dumps(plot_settings, indent=2), encoding='utf-8')
print(f'SHAP samples: {len(sample_indices)}; displayed predictors: {len(display_names)}')

In [ ]:
bootstrap_config = {
    'n_bootstrap': N_BOOTSTRAP, 'sampling_seed': SAMPLING_SEED,
    'model_seed': MODEL_SEED, 'n_estimators': N_ESTIMATORS,
    'cluster_threshold': CLUSTER_THRESHOLD, 'selected_predictors': selected,
    'input_sha256': hashlib.sha256((INPUT_DIR / 'RF_SHAP_samples.csv').read_bytes()).hexdigest(),
    'settings_sha256': hashlib.sha256((INPUT_DIR / 'RF_SHAP_settings.json').read_bytes()).hexdigest()
}
checkpoint_path = OUTPUT_DIR / 'bootstrap_checkpoint.json'
rows_path = OUTPUT_DIR / 'bootstrap_replicates.csv'
if checkpoint_path.is_file():
    checkpoint = json.loads(checkpoint_path.read_text(encoding='utf-8'))
    if checkpoint['config'] != bootstrap_config:
        raise ValueError('Bootstrap settings or input data changed. Use a new output directory.')
    completed = int(checkpoint['completed'])
    saved = pd.read_csv(rows_path, float_precision='round_trip')
    rows = saved.loc[saved['repeat'] <= completed].to_dict('records')
    if len(rows) != completed * len(selected):
        raise ValueError('Bootstrap checkpoint rows are incomplete.')
else:
    completed = 0
    rows = []
rng = np.random.default_rng(SAMPLING_SEED)
for _ in range(completed):
    rng.choice(len(y), size=len(y), replace=True)
for repeat in range(completed + 1, N_BOOTSTRAP + 1):
    indices = rng.choice(len(y), size=len(y), replace=True)
    importance = fit_forest(X[indices], y[indices]).feature_importances_
    ordering = np.argsort(-importance, kind='stable')
    ranks = np.empty(len(selected), dtype=int)
    ranks[ordering] = np.arange(1, len(selected) + 1)
    for index, variable in enumerate(selected):
        rows.append({'repeat': repeat, 'variable': variable, 'rank': int(ranks[index]), 'importance': float(importance[index])})
    if repeat % 50 == 0 or repeat == N_BOOTSTRAP:
        temporary = rows_path.with_suffix('.tmp.csv')
        pd.DataFrame(rows).to_csv(temporary, index=False)
        temporary.replace(rows_path)
        temporary = checkpoint_path.with_suffix('.tmp.json')
        temporary.write_text(json.dumps({'config': bootstrap_config, 'completed': repeat}, indent=2), encoding='utf-8')
        temporary.replace(checkpoint_path)
        print(f'Bootstrap: {repeat}/{N_BOOTSTRAP}')
replicates = pd.DataFrame(rows)
if len(replicates) != N_BOOTSTRAP * len(selected):
    raise AssertionError('Unexpected number of bootstrap records.')
summary = importance_table.rename(columns={'rank': 'original_rank', 'importance': 'original_importance'}).copy()
statistics = replicates.groupby('variable', sort=False).agg(
    top1_count=('rank', lambda values: int((values == 1).sum())),
    top3_count=('rank', lambda values: int((values <= 3).sum())),
    median_rank=('rank', 'median'), mean_importance=('importance', 'mean')
).reset_index()
summary = summary.merge(statistics, on='variable', how='left')
summary['top3_frequency'] = summary.top3_count / N_BOOTSTRAP
summary['retained_60pct'] = (summary.original_rank <= 3) & (summary.top3_frequency >= STABILITY_THRESHOLD)
summary.to_csv(OUTPUT_DIR / 'bootstrap_stability.csv', index=False)
metadata = {
    'settings': SETTINGS, 'n_samples': len(y), 'selected_predictors': selected,
    'n_estimators': N_ESTIMATORS, 'model_seed': MODEL_SEED,
    'training_r2': model.score(X, y), 'shap_baseline': baseline,
    'shap_sample_size': len(sample_indices), 'bootstrap': bootstrap_config,
    'importance_definition': 'Random-forest mean decrease in impurity',
    'stability_rule': 'Retain original top-three predictors with top-three frequency at least 0.60'
}
(OUTPUT_DIR / 'analysis_summary.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print(summary.loc[summary.original_rank <= 3, ['variable', 'top3_frequency', 'retained_60pct']].to_string(index=False))